# Multi-Query Retrieval
질문 하나를 여러 개의 검색 쿼리로 바꾼 뒤, 각 쿼리로 검색한 결과를 합쳐 최종 결과를 만든다.

## 환경설정

In [14]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [15]:
import pandas as pd

document_df = pd.read_csv("data/documents.csv")
queries_df = pd.read_csv("data/queries.csv")
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=2
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=2
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=2


## 검색기 준비

In [16]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

okt = Okt()
tokenized_docs = [okt.morphs(content) for content in document_df['content']]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    """
    BM25로 질문과 관련 있는 상위 문서 ID를 반환한다.
    """
    query_token = okt.morphs(query)
    scores = bm25.get_scores(query_token)
    sorted_idx = sorted(range(len(scores)), key=lambda i:scores[i], reverse=True)
    ranked_docs = [document_df['doc_id'].iloc[i] for i in sorted_idx[:top_k]]
    return ranked_docs

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

def dense_search(query,top_k=5):
    """Dense retrieval로 질문과 관련있는 상위 문서 ID를 반환한다."""
    docs = vector_store.similarity_search(query,k=top_k)
    return [doc.metadata["doc_id"] for doc in docs]

## Multi-Query 생성 체인 만들기

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

multi_query_prompt = PromptTemplate.from_template('''
아래 사용자 질문을 벡터 검색에 사용할 쿼리 3개로 변환하세요.
                                                  
규칙 :
- 원본 질문의 의도를 유지하세요.
- 서로 다른 표현과 관점으로 작성하세요.
- 답변을 작성하지 말고 검색 쿼리만 작성하세요.
- 각 줄에는 검색 쿼리 하나만 작성하세요.
- 번호, 따옴표, 설명 문장은 쓰지 마세요.
                                                  
사용자 질문:
{query}
''')

query_llm = ChatOpenAI(model=OPENAI_LLM_MODEL,temperature=0.3)
output_parser = StrOutputParser()

multi_query_chain = multi_query_prompt | query_llm | output_parser

## 생성된 검색 쿼리 확인

In [ ]:
def parse_multi_queries(text):
    """LLM이 생성한 여러 줄의 검색 쿼리를 리스트로 변환한다."""

    queries = []

    for line in text.splitlines():
        line = line.strip()

        if line:
            queries.append(line)
    
    return queries

def generate_multi_queries(query, include_original=True):
    """원본 질문을 여러 검색 쿼리로 확장한다."""
    response = multi_query_chain.invoke({'query':query})
    generated_queries = parse_multi_queries(response)

    queries = []
    if include_original:
        queries.append(query)

    for generated_query in generated_queries:
        if generated_query not in query:
            queries.append(generated_query)
    
    return queries

In [ ]:
sample_query = queries_df.loc[0,'query_text']
sample_multi_queries = generate_multi_queries(sample_query)

print("원본 질문")
print(sample_query)

print("검색에 사용할 질문")
for idx, query in enumerate(sample_multi_queries, start=1):
    print(f"{idx}. {query}")

## 여러 검색 결과 결합하기
- 단순히 모든 결과를 이어붙이면 중복 문서가 생기고, 어떤 문서를 더 앞에 둘지 기준이 불명확해진다.
- 앞선 RRF 실습에서 사용한 것처럼 순위 기반 결합을 활용해 본다.

In [ ]:
def rrf_rank(*ranked_lists, k=60):
    """여러 검색 결과 리스트를 RRF 점수로 결합한다."""

    candidate_scores = {}

    for ranked_list in ranked_lists:
        for rank, doc_id in enumerate(ranked_list,start=1):
            candidate_scores[doc_id] = candidate_scores.get(doc_id,0) + 1 / (k+rank)

    ranked = sorted(candidate_scores.items(),key=lambda x:x[1], reverse=True)
    return [doc_id for doc_id,score in ranked]

# def multi_query_search(query,query_top_k=10,final_top_k=5):
#     """Multi-Query Retrieval로 최종 문서 ID 목록을 반환한다."""
#     queries = generate_multi_queries(query)
#     ranked_lists = []

#     for search_query in queries:
#         ranked_lists.append(dense_search(search_query,top_k=query_top_k))

#     fused = rrf_rank(*ranked_lists)
#     return fused[:final_top_k]

In [ ]:
sample_ranked_lists = [dense_search(query,top_k=10) for query in sample_multi_queries]
sample_multi_query_result = rrf_rank(*sample_ranked_lists)

comparison_df = pd.DataFrame({
    'rank':range(1,6),
    'Dense':dense_search(sample_query,top_k=5),
    'MultiQuery' : sample_multi_query_result[:5]
})

comparison_df

## 전체 질문에 대해 Multi-Query Retrieval 실행

In [ ]:
from tqdm import tqdm

dense_results = {}
for id,row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    dense_results[qid] = dense_search(query_text,top_k=5)

multi_queries = {}
multi_query_results = {}

for idx,row in tqdm(queries_df.iterrows(),total=len(queries_df)):
    qid = row['query_id']
    query_text = row['query_text']

    queries = generate_multi_queries(query_text)
    multi_queries[qid] = queries

    ranked_lists = []
    for search_query in queries:
        ranked_lists.append(dense_search(search_query, top_k=10))

    multi_query_results[qid] = rrf_rank(*ranked_lists)[:5]

## 생성된 쿼리와 검색 결과 미리보기

In [ ]:
pd.set_option('display.max_colwidth',None)

preview_rows = []

for qid in queries_df['query_id'].head(5):
    query_text = queries_df.loc[queries_df['query_id'] == qid, 'query_text'].iloc[0],
    preview_rows.append({
        'query_id':qid,
        'query_text':query_text,
        'generated_queries': multi_queries[qid],
        'Dense' : dense_results[qid],
        'Multi_Query':multi_query_results[qid]
    })

pd.DataFrame(preview_rows)

# 평가 함수

In [12]:
import numpy as np

def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())
    top_k = predicted[:k]
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break
    num_correct = 0
    precision_sum = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0
    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]
        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)
    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

In [13]:
dense_metrics =  evaluate_all(dense_results,queries_df)
multi_query_metrics = evaluate_all(multi_query_results,queries_df)

metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recall@k","MRR","MAP"],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
    'Multi-Query' : [multi_query_metrics["Precision@k"],multi_query_metrics["Recall@k"],multi_query_metrics["MRR"],multi_query_metrics["MAP"]],
})
metrics_df

,Metric,Dense,Multi-Query
0,Precision@k,0.233333,0.240000
1,Recall@k,0.975000,0.983333
2,MRR,1.000000,1.000000
3,MAP,0.975000,0.981667
